<a href="https://colab.research.google.com/github/lce2005/2026_OIBC/blob/leece/exp04_catboost_gpu_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!nvidia-smi

Thu Nov  5 14:35:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             31W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
%pip install -q catboost pyarrow

In [10]:
import catboost
from catboost.utils import get_gpu_device_count

print("CatBoost 버전:", catboost.__version__)
print("사용 가능한 GPU 개수:", get_gpu_device_count())

CatBoost 버전: 1.2.10
사용 가능한 GPU 개수: 1


In [11]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Data/data_v0.4")

TRAIN_PATH = DATA_DIR / "train_v0.4.parquet"
TEST_PATH = DATA_DIR / "test_v0.4.parquet"
DICT_PATH = DATA_DIR / "data_dictionary_v0.4.csv"

for path in [TRAIN_PATH, TEST_PATH, DICT_PATH]:
    print(path.name, "→", "확인 완료" if path.is_file() else "찾지 못함")

train_v0.4.parquet → 확인 완료
test_v0.4.parquet → 확인 완료
data_dictionary_v0.4.csv → 확인 완료


In [12]:
import gc
import pandas as pd
from pathlib import Path
from google.colab import drive

# 구글 드라이브 자동 마운트
try:
    drive.mount("/content/drive")
except Exception as e:
    print("드라이브 마운트 중 오류 발생:", e)

TRAIN_PATH = Path(
    "/content/drive/MyDrive/Data/data_v0.4/train_v0.4.parquet"
)

# 1. 학습 데이터 읽기
df = pd.read_parquet(TRAIN_PATH)
df["ts"] = pd.to_datetime(df["ts"], utc=True).dt.tz_convert("Asia/Seoul")

print("전체 데이터 크기:", df.shape)

# 2. 팀 합의 구간(v0.4)의 split 열 사용
# train: 2023.06~2025.02 / dev: 2025.03~2025.04 / holdout: 2025.05
assert "split" in df.columns, "v0.4 데이터에 split 열이 없습니다."

train_df = df.loc[df["split"].eq("train")].copy()
dev_df = df.loc[df["split"].eq("dev")].copy()
holdout_df = df.loc[df["split"].eq("holdout")].copy()

del df
gc.collect()

# 4. 분할 결과 확인
assert not train_df.empty, "학습 데이터가 비어 있습니다."
assert not dev_df.empty, "검증 데이터가 비어 있습니다."
assert train_df["ts"].max() < dev_df["ts"].min()
assert train_df["cf_target"].notna().all(), "학습 정답에 결측이 있습니다."

print(f"\n학습 데이터: {len(train_df):,}행")
print(f"검증 데이터: {len(dev_df):,}행")
print("학습 기간:", train_df["ts"].min(), "~", train_df["ts"].max())
print("검증 기간:", dev_df["ts"].min(), "~", dev_df["ts"].max())

display(
    train_df[
        ["site_id", "ts", "capacity_kw", "energy_kwh", "cf_target"]
    ].head()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 데이터 크기: (4392566, 100)

학습 데이터: 2,868,616행
검증 데이터: 370,392행
학습 기간: 2023-06-01 01:00:00+09:00 ~ 2025-03-01 00:00:00+09:00
검증 기간: 2025-03-01 01:00:00+09:00 ~ 2025-05-01 00:00:00+09:00


,site_id,ts,capacity_kw,energy_kwh,cf_target
0,PV001,2024-02-01 12:00:00+09:00,99.959999,6.0,0.060024
1,PV001,2024-02-01 13:00:00+09:00,99.959999,12.0,0.120048
2,PV001,2024-02-01 14:00:00+09:00,99.959999,15.0,0.150060
3,PV001,2024-02-01 15:00:00+09:00,99.959999,12.0,0.120048
4,PV001,2024-02-01 16:00:00+09:00,99.959999,6.0,0.060024


In [13]:
# 앞에서 만든 train_df, dev_df, TRAIN_PATH가 있는 상태에서 실행하세요.
# CatBoost 2종 학습 → 검증으로 조합 선택 → 별도 5월 평가 → Drive 저장

import gc
import json
import time
import uuid
import numpy as np
import pandas as pd
import catboost

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from catboost import CatBoostRegressor, Pool
from catboost.utils import get_gpu_device_count

assert get_gpu_device_count() > 0, "Colab GPU 연결을 확인하세요."

SEED = 42
EXP_ID = "exp04_catboost_gpu"
RUN_ID = (
    datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")
    + "_" + uuid.uuid4().hex[:6]
)
SAVE_DIR = Path("/content/drive/MyDrive/OIBC_experiments") / EXP_ID / RUN_ID
SAVE_DIR.mkdir(parents=True, exist_ok=False)

print("결과 저장 위치:", SAVE_DIR)

# ============================================================
# 1. 기존 XGBoost와 같은 기본 입력 44개
# ============================================================
CAT_FEATURES = ["site_id", "wa_station", "wb_station"]

NUM_FEATURES = [
    "capacity_kw", "lat", "lon", "wa_dist", "wb_dist",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
    "elev", "azim", "cosz", "is_daylight",
    "wa_cloud_r10", "wa_cloud_r17",
    "wa_temp_r10", "wa_temp_r17",
    "wa_humidity_r10", "wa_humidity_r17",
    "wa_wind_speed_r10", "wa_wind_speed_r17",
    "wa_rain_r10", "wa_rain_r17",
    "wb_cloud_r10", "wb_cloud_r17",
    "wb_temp_r10", "wb_temp_r17",
    "wb_rel_hum_r10", "wb_rel_hum_r17",
    "wb_uv_idx_r10", "wb_uv_idx_r17",
    "wb_precip_prob_r10", "wb_precip_prob_r17",
    "wb_total_liq_r10", "wb_total_liq_r17",
    "wb_wind_speed_ms_r10", "wb_wind_speed_ms_r17",
    "fc_filled_a", "fc_filled_b", "fc_missing_a", "fc_missing_b",
]
BASE_FEATURES = CAT_FEATURES + NUM_FEATURES
assert len(BASE_FEATURES) == 44


# ============================================================
# 2. 예보 변화·제공사 차이·태양 위치와 구름의 조합
#    실제 발전량과 정답은 입력 특징에 사용하지 않습니다.
# ============================================================
def make_features(frame):
    x = frame[BASE_FEATURES].copy()

    for col in CAT_FEATURES:
        x[col] = frame[col].astype("string").fillna("__MISSING__").astype(str)

    for col in NUM_FEATURES:
        x[col] = pd.to_numeric(x[col], errors="raise").astype("float32")

    # 전일 10시 예보와 17시 예보의 평균·변화량
    weather_pairs = [
        "wa_cloud", "wa_temp", "wa_humidity",
        "wa_wind_speed", "wa_rain",
        "wb_cloud", "wb_temp", "wb_rel_hum",
        "wb_uv_idx", "wb_precip_prob",
        "wb_total_liq", "wb_wind_speed_ms",
    ]

    for prefix in weather_pairs:
        a = x[f"{prefix}_r10"]
        b = x[f"{prefix}_r17"]
        x[f"{prefix}_mean"] = (a + b) / 2
        x[f"{prefix}_revision"] = b - a

    # 같은 물리량에 대한 A사·B사 예보 차이
    for label, a, b in [
        ("cloud", "wa_cloud", "wb_cloud"),
        ("temp", "wa_temp", "wb_temp"),
        ("humidity", "wa_humidity", "wb_rel_hum"),
        ("wind", "wa_wind_speed", "wb_wind_speed_ms"),
    ]:
        delta = x[f"{a}_mean"] - x[f"{b}_mean"]
        x[f"ab_{label}_diff"] = delta
        x[f"ab_{label}_disagreement"] = delta.abs()

    # 계산된 태양 위치와 예보 운량을 결합한 보조 특징
    cosz = x["cosz"].clip(lower=0)
    for provider in ["wa", "wb"]:
        cloud = x[f"{provider}_cloud_mean"].clip(0, 100) / 100
        x[f"{provider}_sun_cloud_proxy"] = cosz * (1 - cloud)
        x[f"{provider}_sun_cloud_squared"] = cosz * (1 - cloud) ** 2

    numeric = [col for col in x.columns if col not in CAT_FEATURES]
    x[numeric] = (
        x[numeric]
        .replace([np.inf, -np.inf], np.nan)
        .astype("float32")
    )
    return x


def validate_frame(frame, label):
    required = BASE_FEATURES + ["ts", "cf_target", "energy_kwh"]
    missing = sorted(set(required) - set(frame.columns))
    assert not missing, f"{label}: 필요한 열이 없습니다: {missing}"
    assert len(frame) > 0, f"{label}: 데이터가 없습니다."
    assert not frame.duplicated(["site_id", "ts"]).any(), f"{label}: 중복 행"

    for col in ["cf_target", "energy_kwh", "capacity_kw"]:
        assert np.isfinite(frame[col].to_numpy(dtype=float)).all(), (
            f"{label}: {col}에 결측 또는 무한대가 있습니다."
        )
    assert (frame["capacity_kw"] > 0).all()
    assert (frame["energy_kwh"] >= 0).all()
    assert frame["is_daylight"].isin([0, 1]).all()


validate_frame(train_df, "학습")
validate_frame(dev_df, "검증")
assert train_df["ts"].max() < dev_df["ts"].min()

# 야간 예측은 0으로 고정하므로 모델은 주간 데이터로 학습합니다.
train_day = train_df["is_daylight"].eq(1)
dev_day = dev_df["is_daylight"].eq(1)
assert train_day.any() and dev_day.any()

X_train = make_features(train_df.loc[train_day])
X_dev = make_features(dev_df.loc[dev_day])
FEATURES = X_train.columns.tolist()

y_train = train_df.loc[train_day, "cf_target"].to_numpy(dtype=np.float32)
y_dev = dev_df.loc[dev_day, "cf_target"].to_numpy(dtype=np.float32)

cap_train = train_df.loc[train_day, "capacity_kw"].to_numpy(dtype=np.float64)
cap_dev = dev_df.loc[dev_day, "capacity_kw"].to_numpy(dtype=np.float64)

# cf 오차에 설비용량을 곱하면 kWh 오차가 됩니다.
# 검증 MAE도 설비용량으로 가중하여 kWh 오차와 같은 방향으로 선택합니다.
dev_pool = Pool(
    X_dev,
    label=y_dev,
    cat_features=CAT_FEATURES,
    weight=(cap_dev / cap_dev.mean()).astype("float32"),
)

print(f"입력 변수: {len(FEATURES)}개")
print(f"주간 학습: {len(X_train):,}행")
print(f"주간 검증: {len(X_dev):,}행")


# ============================================================
# 3. 절대오차 모델 + 큰 오차를 더 줄이는 제곱오차 모델
# ============================================================
CONFIGS = {
    "cat_mae": {"loss_function": "MAE", "depth": 7, "weight_power": 1},
    "cat_rmse": {"loss_function": "RMSE", "depth": 7, "weight_power": 2},
}

models = {}
training_info = {}

for name, config in CONFIGS.items():
    print(f"\n{'=' * 20} {name} 학습 {'=' * 20}")

    weights = (cap_train / cap_train.mean()) ** config["weight_power"]
    weights = (weights / weights.mean()).astype("float32")

    train_pool = Pool(
        X_train,
        label=y_train,
        cat_features=CAT_FEATURES,
        weight=weights,
    )

    model = CatBoostRegressor(
        loss_function=config["loss_function"],
        eval_metric="MAE",
        iterations=1600,
        learning_rate=0.04,
        depth=config["depth"],
        l2_leaf_reg=8.0,
        random_seed=SEED,
        task_type="GPU",
        devices="0",
        gpu_ram_part=0.80,
        border_count=128,
        max_ctr_complexity=1,
        boosting_type="Plain",
        bootstrap_type="Bayesian",
        bagging_temperature=1.0,
        thread_count=4,
        allow_writing_files=False,
    )

    started = time.perf_counter()
    model.fit(
        train_pool,
        eval_set=dev_pool,
        early_stopping_rounds=120,
        use_best_model=True,
        verbose=100,
    )

    model.save_model(str(SAVE_DIR / f"{name}.cbm"))
    models[name] = model
    training_info[name] = {
        "best_iteration": int(model.get_best_iteration()),
        "tree_count": int(model.tree_count_),
        "training_seconds": round(time.perf_counter() - started, 2),
        "weight_power": config["weight_power"],
        "parameters": model.get_all_params(),
    }

    pd.DataFrame({
        "feature": FEATURES,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).to_csv(
        SAVE_DIR / f"{name}_feature_importance.csv", index=False
    )

    del train_pool
    gc.collect()

del X_train, X_dev, dev_pool
gc.collect()


# ============================================================
# 4. 발전소별 kWh 예측과 평가 함수
# ============================================================
def predict_kwh(model, frame, batch_size=100_000):
    result = np.zeros(len(frame), dtype=np.float64)
    daylight_positions = np.flatnonzero(
        frame["is_daylight"].to_numpy() == 1
    )

    for start in range(0, len(daylight_positions), batch_size):
        positions = daylight_positions[start:start + batch_size]
        batch = frame.iloc[positions]
        x = make_features(batch)
        assert x.columns.tolist() == FEATURES

        cf = model.predict(Pool(x, cat_features=CAT_FEATURES))
        result[positions] = (
            np.maximum(cf, 0.0)
            * batch["capacity_kw"].to_numpy(dtype=np.float64)
        )

    assert np.isfinite(result).all() and (result >= 0).all()
    return result


def metrics(frame, prediction):
    actual = frame["energy_kwh"].to_numpy(dtype=np.float64)
    error = prediction - actual
    day = frame["is_daylight"].to_numpy() == 1

    result = {
        "rows": int(len(frame)),
        "mae_kwh": float(np.abs(error).mean()),
        "rmse_kwh": float(np.sqrt(np.mean(error ** 2))),
        "daylight_mae_kwh": float(np.abs(error[day]).mean()),
        "bias_kwh": float(error.mean()),
    }

    if {"smp", "k"}.issubset(frame.columns):
        weight = frame["smp"].to_numpy(float) * frame["k"].to_numpy(float)
        assert np.isfinite(weight).all() and (weight >= 0).all()
        result["site_weighted_penalty_proxy"] = float(
            0.10 * np.sum(weight * np.abs(error))
        )
    return result


def save_predictions(frame, prediction, prefix):
    output = frame[["site_id", "ts", "energy_kwh"]].copy()
    output = output.rename(columns={"energy_kwh": "actual_kwh"})
    output["pred_kwh"] = prediction
    output["error_kwh"] = output["pred_kwh"] - output["actual_kwh"]
    output = output.sort_values(["site_id", "ts"])

    output.to_csv(SAVE_DIR / f"{prefix}_errors.csv", index=False)
    output[["site_id", "ts", "pred_kwh"]].to_csv(
        SAVE_DIR / f"{prefix}_predictions.csv", index=False
    )


# ============================================================
# 5. 3~4월 검증 데이터로 두 모델의 조합 선택
# ============================================================
dev_mae_pred = predict_kwh(models["cat_mae"], dev_df)
dev_rmse_pred = predict_kwh(models["cat_rmse"], dev_df)

comparison = []
for mae_weight in [0.0, 0.25, 0.50, 0.75, 1.0]:
    prediction = (
        mae_weight * dev_mae_pred
        + (1 - mae_weight) * dev_rmse_pred
    )
    comparison.append({
        "mae_model_weight": mae_weight,
        **metrics(dev_df, prediction),
    })

comparison = pd.DataFrame(comparison).sort_values(
    ["mae_kwh", "rmse_kwh"]
).reset_index(drop=True)

display(comparison[[
    "mae_model_weight", "mae_kwh", "rmse_kwh",
    "daylight_mae_kwh", "bias_kwh",
]])

# 선택 기준은 발전소별 kWh MAE입니다.
# 그룹 배정 데이터가 없으므로 공식 그룹 정산 페널티는 계산하지 않습니다.
BEST_WEIGHT = float(comparison.iloc[0]["mae_model_weight"])
dev_prediction = (
    BEST_WEIGHT * dev_mae_pred
    + (1 - BEST_WEIGHT) * dev_rmse_pred
)

comparison.to_csv(SAVE_DIR / "dev_blend_comparison.csv", index=False)
save_predictions(dev_df, dev_mae_pred, "cat_mae_dev")
save_predictions(dev_df, dev_rmse_pred, "cat_rmse_dev")
save_predictions(dev_df, dev_prediction, "selected_dev")

run_info = {
    "experiment": EXP_ID,
    "run_id": RUN_ID,
    "catboost_version": catboost.__version__,
    "data_path": str(TRAIN_PATH),
    "seed": SEED,
    "features": FEATURES,
    "categorical_features": CAT_FEATURES,
    "train_rows_all": int(len(train_df)),
    "train_rows_daylight": int(train_day.sum()),
    "dev_rows_all": int(len(dev_df)),
    "train_first_ts": str(train_df["ts"].min()),
    "train_last_ts": str(train_df["ts"].max()),
    "dev_first_ts": str(dev_df["ts"].min()),
    "dev_last_ts": str(dev_df["ts"].max()),
    "training": training_info,
    "selected_mae_model_weight": BEST_WEIGHT,
    "selection_metric": "development site-level MAE in kWh",
    "dev_metrics": metrics(dev_df, dev_prediction),
    "notes": [
        "Daylight-only training; nighttime predictions fixed to zero.",
        "MAE model uses capacity weights; RMSE model uses capacity squared weights.",
        "GPU training is not bitwise deterministic.",
        "Site-weighted penalty proxy is not the official group penalty.",
        "Development results were used for model and blend selection.",
        "These are development models, not models refitted on all available history.",
    ],
}

with open(SAVE_DIR / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)

print(f"\n선택한 조합: MAE 모델 {BEST_WEIGHT:.0%} + RMSE 모델 {1-BEST_WEIGHT:.0%}")
print(f"3~4월 검증 MAE: {run_info['dev_metrics']['mae_kwh']:.4f} kWh")


# ============================================================
# 6. 조합을 고정한 뒤, 학습·선택에 쓰지 않은 5월에서 확인
#    5월 결과를 보고 설정을 바꾸면 이후에는 독립 평가가 아닙니다.
# ============================================================
HOLD_START = pd.Timestamp("2025-05-01 01:00", tz="Asia/Seoul")
HOLD_END = pd.Timestamp("2025-06-01 01:00", tz="Asia/Seoul")
assert dev_df["ts"].max() < HOLD_START

hold_columns = BASE_FEATURES + ["ts", "cf_target", "energy_kwh", "smp", "k"]

hold_df = pd.read_parquet(
    TRAIN_PATH,
    columns=hold_columns,
    filters=[
        ("ts", ">=", HOLD_START.to_pydatetime()),
        ("ts", "<", HOLD_END.to_pydatetime()),
    ],
)
hold_df["ts"] = pd.to_datetime(
    hold_df["ts"], utc=True
).dt.tz_convert("Asia/Seoul")

validate_frame(hold_df, "5월 평가")
assert hold_df["ts"].ge(HOLD_START).all()
assert hold_df["ts"].lt(HOLD_END).all()

hold_prediction = np.zeros(len(hold_df), dtype=np.float64)
if BEST_WEIGHT > 0:
    hold_prediction += BEST_WEIGHT * predict_kwh(models["cat_mae"], hold_df)
if BEST_WEIGHT < 1:
    hold_prediction += (1 - BEST_WEIGHT) * predict_kwh(
        models["cat_rmse"], hold_df
    )

hold_metrics = metrics(hold_df, hold_prediction)
save_predictions(hold_df, hold_prediction, "selected_holdout_may")

run_info["holdout_metrics"] = hold_metrics
run_info["holdout_first_ts"] = str(hold_df["ts"].min())
run_info["holdout_last_ts"] = str(hold_df["ts"].max())

with open(SAVE_DIR / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)

print("\n========== 학습 및 저장 완료 ==========")
print(f"3~4월 검증 MAE: {run_info['dev_metrics']['mae_kwh']:.4f} kWh")
print(f"5월 별도 평가 MAE: {hold_metrics['mae_kwh']:.4f} kWh")
print(f"5월 별도 평가 RMSE: {hold_metrics['rmse_kwh']:.4f} kWh")
print("두 기간은 날씨와 발전량 분포가 달라 점수를 직접 우열 비교하지 않습니다.")
print("모델·예측값·실험 기록 저장 위치:", SAVE_DIR)

결과 저장 위치: /content/drive/MyDrive/OIBC_experiments/exp04_catboost_gpu/20261105_233549_5367e7
입력 변수: 80개
주간 학습: 1,411,930행
주간 검증: 190,721행

==================== cat_mae 학습 ====================


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.2019535	test: 0.2365434	best: 0.2365434 (0)	total: 70.6ms	remaining: 1m 52s
100:	learn: 0.0738366	test: 0.0741577	best: 0.0741577 (100)	total: 4.64s	remaining: 1m 8s
200:	learn: 0.0698530	test: 0.0703944	best: 0.0703944 (200)	total: 10.4s	remaining: 1m 12s
300:	learn: 0.0676025	test: 0.0693488	best: 0.0693488 (300)	total: 15.2s	remaining: 1m 5s
400:	learn: 0.0659964	test: 0.0691955	best: 0.0688522 (362)	total: 19.8s	remaining: 59.3s
bestTest = 0.06885218418
bestIteration = 362
Shrink model to first 363 iterations.

==================== cat_rmse 학습 ====================


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.1976160	test: 0.2341705	best: 0.2341705 (0)	total: 51.7ms	remaining: 1m 22s
100:	learn: 0.0718717	test: 0.0805351	best: 0.0805351 (100)	total: 4.38s	remaining: 1m 5s
200:	learn: 0.0663689	test: 0.0768810	best: 0.0768289 (198)	total: 8.07s	remaining: 56.2s
300:	learn: 0.0637094	test: 0.0757942	best: 0.0757870 (298)	total: 11.2s	remaining: 48.5s
400:	learn: 0.0617734	test: 0.0753598	best: 0.0753290 (383)	total: 14.3s	remaining: 42.8s
500:	learn: 0.0603154	test: 0.0750053	best: 0.0750053 (500)	total: 17.9s	remaining: 39.4s
600:	learn: 0.0591197	test: 0.0748791	best: 0.0748170 (575)	total: 22.5s	remaining: 37.4s
700:	learn: 0.0579974	test: 0.0747481	best: 0.0747481 (700)	total: 25.5s	remaining: 32.7s
800:	learn: 0.0569983	test: 0.0746356	best: 0.0746165 (795)	total: 28.6s	remaining: 28.5s
900:	learn: 0.0560955	test: 0.0745988	best: 0.0745956 (851)	total: 32s	remaining: 24.9s
1000:	learn: 0.0552691	test: 0.0746959	best: 0.0745897 (922)	total: 36.2s	remaining: 21.7s
bestTest = 0.

,mae_model_weight,mae_kwh,rmse_kwh,daylight_mae_kwh,bias_kwh
0,1.00,5.995672,23.278067,11.598395,0.611680
1,0.75,6.041058,23.462921,11.686538,0.184839
2,0.50,6.139050,24.353681,11.876845,-0.242001
3,0.25,6.282051,25.877553,12.154562,-0.668842
4,0.00,6.463721,27.931106,12.507376,-1.095683



선택한 조합: MAE 모델 100% + RMSE 모델 0%
3~4월 검증 MAE: 5.9957 kWh

========== 학습 및 저장 완료 ==========
3~4월 검증 MAE: 5.9957 kWh
5월 별도 평가 MAE: 6.9758 kWh
5월 별도 평가 RMSE: 25.0273 kWh
두 기간은 날씨와 발전량 분포가 달라 점수를 직접 우열 비교하지 않습니다.
모델·예측값·실험 기록 저장 위치: /content/drive/MyDrive/OIBC_experiments/exp04_catboost_gpu/20261105_233549_5367e7


In [14]:
import json

# 실험 결과가 저장된 run_info.json 로드
run_info_path = SAVE_DIR / 'run_info.json'

try:
    with open(run_info_path, 'r', encoding='utf-8') as f:
        info = json.load(f)

    # 주요 정보 요약 출력
    print("=== [실험 기본 정보] ===")
    print(f"실험 ID: {info.get('experiment')}")
    print(f"런 ID: {info.get('run_id')}")
    print(f"학습에 사용된 특징 수: {len(info.get('features', []))}개")

    print("\n=== [모델 학습 요약] ===")
    for model_name, details in info.get('training', {}).items():
        print(f"\n▶ {model_name}:")
        print(f"  - 최적 이터레이션 (Best Iteration): {details.get('best_iteration')}")
        print(f"  - 트리 개수: {details.get('tree_count')}")
        print(f"  - 학습 소요 시간: {details.get('training_seconds')}초")

    print("\n=== [앙상블/블렌딩 비율] ===")
    mae_w = info.get('selected_mae_model_weight', 0)
    print(f"선택된 조합: cat_mae ({mae_w:.0%}) + cat_rmse ({1-mae_w:.0%})")

    print("\n=== [평가 지표 (Metrics)] ===")
    print("▶ 3~4월 검증 데이터 (dev_metrics):")
    for k, v in info.get('dev_metrics', {}).items():
        print(f"  - {k}: {v:.6f}" if isinstance(v, float) else f"  - {k}: {v}")

    if 'holdout_metrics' in info:
        print("\n▶ 5월 별도 평가 데이터 (holdout_metrics):")
        for k, v in info.get('holdout_metrics', {}).items():
            print(f"  - {k}: {v:.6f}" if isinstance(v, float) else f"  - {k}: {v}")
    else:
        print("\n※ 5월 별도 평가(holdout_metrics) 결과는 아직 run_info.json에 기록되지 않았습니다.")

except FileNotFoundError:
    print(f"오류: {run_info_path} 파일을 찾을 수 없습니다. 학습 셀이 끝까지 완료되었는지 확인해 주세요.")
except Exception as e:
    print("오류 발생:", e)

=== [실험 기본 정보] ===
실험 ID: exp04_catboost_gpu
런 ID: 20261105_233549_5367e7
학습에 사용된 특징 수: 80개

=== [모델 학습 요약] ===

▶ cat_mae:
  - 최적 이터레이션 (Best Iteration): 362
  - 트리 개수: 363
  - 학습 소요 시간: 33.18초

▶ cat_rmse:
  - 최적 이터레이션 (Best Iteration): 922
  - 트리 개수: 923
  - 학습 소요 시간: 44.2초

=== [앙상블/블렌딩 비율] ===
선택된 조합: cat_mae (100%) + cat_rmse (0%)

=== [평가 지표 (Metrics)] ===
▶ 3~4월 검증 데이터 (dev_metrics):
  - rows: 370392
  - mae_kwh: 5.995672
  - rmse_kwh: 23.278067
  - daylight_mae_kwh: 11.598395
  - bias_kwh: 0.611680
  - site_weighted_penalty_proxy: 57134374.430895

▶ 5월 별도 평가 데이터 (holdout_metrics):
  - rows: 188232
  - mae_kwh: 6.975762
  - rmse_kwh: 25.027264
  - daylight_mae_kwh: 12.066403
  - bias_kwh: 0.507658
  - site_weighted_penalty_proxy: 35808942.639933
